# 🚀 Triad-Omni v5.1: LoRA Trainer with Full Versioning (Google Colab Tesla T4)
### Автономное обучение адаптеров личности и инженерного стиля для Xiaomi Mi 10T 5G (Snapdragon 865)
- **Базовая модель:** `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (5.5 ГБ VRAM)
- **Аппаратный ускоритель:** Tesla T4 GPU (Turing `sm_75`)
- **Точность:** `fp16=True, bf16=False` (T4 аппаратное ускорение fp16)
- **Версионирование картриджей:** `<slug>_<version>_<timestamp>.omni.zip` с паспортом `card.json`
- **Автовыгрузка в облако:** Сохранение истории версий в Google Диск (папки `1UNP2g3jOf...` и `1WJj716R...`) через PyDrive2

In [ ]:
#@title 🎛️ 1. Настройки параметров, Версионирование и Установка зависимостей
LORA_RANK = 16 #@param {type:"slider", min:4, max:64, step:4}
LORA_ALPHA = 16 #@param {type:"slider", min:8, max:64, step:8}
MAX_STEPS = 60 #@param {type:"slider", min:20, max:300, step:10}
LEARNING_RATE = 2e-4 #@param {type:"number"}
MAX_SEQ_LENGTH = 1536 #@param {type:"slider", min:512, max:2048, step:256}
CARTRIDGE_SLUG = "triad-omni-persona" #@param {type:"string"}
CARTRIDGE_VERSION = "v1.0" #@param {type:"string"}
ADD_TIMESTAMP = True #@param {type:"boolean"}

from datetime import datetime, timezone
import os, json, hashlib

# Формируем уникальный идентификатор сборки
TIME_TAG = datetime.now().strftime("%Y%m%d_%H%M")
BUILD_TAG = f"{CARTRIDGE_VERSION}_{TIME_TAG}" if ADD_TIMESTAMP else CARTRIDGE_VERSION
ARCHIVE_FILENAME = f"{CARTRIDGE_SLUG}_{BUILD_TAG}.omni.zip"
LATEST_FILENAME = f"{CARTRIDGE_SLUG}_latest.omni.zip"

print(f"🏷️ Слот картриджа:  {CARTRIDGE_SLUG}")
print(f"📌 Версия сборки:   {BUILD_TAG}")
print(f"📦 Целевой архив:   {ARCHIVE_FILENAME}")

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes pydrive2 oauth2client
import torch
assert torch.cuda.is_available(), "Подключите GPU: Runtime -> Change runtime type -> T4 GPU"
print(f"✅ Графический процессор: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title 📦 2. Загрузка базовой модели Qwen2.5-7B (4-bit) и LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = torch.float16 # Обязательно fp16 под Tesla T4!
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    lora_alpha = LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0.05,
    bias = "none",
    use_rslora = False,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407
)
print("✅ LoRA-слои смонтированы в модель!")

In [ ]:
#@title 📁 3. Загрузка датасета и Старт Обучения
DATASET_URL = "" #@param {type:"string"}
from google.colab import files
import os, shutil, urllib.request, json

target_dataset = "dataset.jsonl"

# Способ А: Загрузка по прямой ссылке или с Яндекс.Диска
if DATASET_URL and DATASET_URL.strip():
    url = DATASET_URL.strip()
    print(f"🌐 Загрузка датасета по ссылке: {url}")
    if "disk.yandex" in url or "yadi.sk" in url:
        api_url = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={url}"
        try:
            with urllib.request.urlopen(api_url) as resp:
                download_data = json.loads(resp.read().decode('utf-8'))
                direct_url = download_data.get("href")
                urllib.request.urlretrieve(direct_url, target_dataset)
                print("✅ Датасет успешно загружен с Яндекс Диска!")
        except Exception as e:
            print(f"⚠️ Ошибка загрузки с Яндекс Диска: {e}. Переходим к ручной загрузке.")
    else:
        try:
            urllib.request.urlretrieve(url, target_dataset)
            print("✅ Датасет успешно загружен по прямой ссылке!")
        except Exception as e:
            print(f"⚠️ Ошибка загрузки по ссылке: {e}. Переходим к ручной загрузке.")

# Способ Б: Ручная загрузка triad_omni_sft_dataset.jsonl если файл еще не готов
if not os.path.exists(target_dataset):
    candidates = [f for f in os.listdir('.') if f.endswith('.jsonl') and f != target_dataset]
    if candidates:
        shutil.copy2(candidates[0], target_dataset)
        print(f"✅ Использован существующий файл датасета: {candidates[0]}")
    else:
        print("📤 Выберите файл triad_omni_sft_dataset.jsonl с вашего ПК:")
        uploaded = files.upload()
        for fname in uploaded.keys():
            shutil.copy2(fname, target_dataset)
            break

from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Форматирование ChatML диалогов
raw_ds = load_dataset("json", data_files=target_dataset)["train"]

def formatting_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_ds = raw_ds.map(formatting_func, batched=True)
print(f"📊 Загружено {len(train_ds)} обучающих сэмплов (накопительный датасет).")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = MAX_STEPS,
        learning_rate = LEARNING_RATE,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        fp16 = True,
        bf16 = False,
        logging_steps = 5,
        output_dir = "outputs",
        report_to = "none"
    )
)

print("🚀 Тренировка запущена! Наблюдайте за падением функции потерь (Loss):")
trainer.train()
print("🎉 Обучение успешно завершено!")

In [ ]:
#@title 💾 4. Конвертация в GGUF LoRA и Скачивание Версионированного Картриджа
import hashlib, json, os, shutil
from datetime import datetime, timezone
from google.colab import files
from huggingface_hub import hf_hub_download

print("⚡ Шаг 1: Сохраняем локальные веса адаптера...")
model.save_pretrained("adapter_files")
tokenizer.save_pretrained("adapter_files")

print("⚙️ Шаг 2: Скачиваем конфигурационные файлы базовой модели Qwen2.5-7B-Instruct...")
base_model_dir = "qwen_base_config"
os.makedirs(base_model_dir, exist_ok=True)

config_files = [
    "config.json",
    "generation_config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.json",
    "merges.txt"
]

for file_name in config_files:
    try:
        hf_hub_download(
            repo_id="Qwen/Qwen2.5-7B-Instruct",
            filename=file_name,
            local_dir=base_model_dir
        )
        print(f"  - Успешно скачан {file_name}")
    except Exception as e:
        print(f"  - Пропуск {file_name} (не критично): {e}")

# Подменяем путь к базовой модели в конфиге адаптера
config_path = "adapter_files/adapter_config.json"
with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = os.path.abspath(base_model_dir)
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("📦 Шаг 3: Проверка конвертера llama.cpp...")
if not os.path.exists("llama.cpp"):
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
    !pip install -q -r llama.cpp/requirements.txt

print("⚙️ Шаг 4: Конвертация LoRA весов в adapter.gguf...")
!python llama.cpp/convert_lora_to_gguf.py adapter_files --base {base_model_dir} --outfile adapter.gguf

if not os.path.exists("adapter.gguf"):
    raise FileNotFoundError("Ошибка: adapter.gguf не был создан! Проверьте логи конвертации выше.")

size_mb = os.path.getsize("adapter.gguf") / 1024 / 1024
print(f"✅ Адаптер готов: adapter.gguf ({size_mb:.1f} МБ)!")

# Считаем SHA256 и формируем паспорт картриджа card.json
with open("adapter.gguf", "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()

now_iso = datetime.now(timezone.utc).isoformat()
card_json = {
    "schema": "omni.lora.card/1",
    "slug": CARTRIDGE_SLUG,
    "version": CARTRIDGE_VERSION,
    "build_tag": BUILD_TAG,
    "created_at": now_iso,
    "rank": LORA_RANK,
    "alpha": LORA_ALPHA,
    "steps": MAX_STEPS,
    "lr": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "sha256": sha,
    "base": "Qwen2.5-7B-Instruct",
    "device_target": "Xiaomi Mi 10T 5G (Snapdragon 865)",
    "description": f"LoRA-картридж Triad-Omni ({CARTRIDGE_SLUG}) {BUILD_TAG}"
}
with open("card.json", "w", encoding="utf-8") as f:
    json.dump(card_json, f, ensure_ascii=False, indent=2)

print(f"📦 Упаковываем версионированный архив: {ARCHIVE_FILENAME}...")
!zip -q "{ARCHIVE_FILENAME}" adapter.gguf card.json

# Создаем копии для совместимости со старыми авто-скриптами
shutil.copy2(ARCHIVE_FILENAME, "cartridge.omni.zip")
shutil.copy2(ARCHIVE_FILENAME, LATEST_FILENAME)

cart_size = os.path.getsize(ARCHIVE_FILENAME) / 1024 / 1024
print(f"🎉 Картридж готов ({cart_size:.1f} МБ)! Начинаем скачивание {ARCHIVE_FILENAME}...")
files.download(ARCHIVE_FILENAME)

In [ ]:
#@title 🔍 5. Проверка паспорта и целостности картриджа
import os, zipfile, json

check_file = ARCHIVE_FILENAME if os.path.exists(ARCHIVE_FILENAME) else "cartridge.omni.zip"
if os.path.exists(check_file):
    size_mb = os.path.getsize(check_file) / 1024 / 1024
    print(f"📦 Проверяемый архив: {check_file} ({size_mb:.2f} МБ)")
    with zipfile.ZipFile(check_file, "r") as z:
        for info in z.infolist():
            fsize = info.file_size / 1024 / 1024
            csize = info.compress_size / 1024 / 1024
            print(f"  - {info.filename:<15} | Исходный: {fsize:.2f} МБ | Сжатый: {csize:.2f} МБ")
        if "card.json" in z.namelist():
            card_data = json.loads(z.read("card.json").decode('utf-8'))
            print("\n📜 Паспорт картриджа (card.json):")
            print(json.dumps(card_data, ensure_ascii=False, indent=2))
else:
    print(f"❌ Файл {check_file} не найден! Сначала выполните шаг 4.")

In [ ]:
#@title ☁️ 6. Автоматическая выгрузка в облако Google Диск с сохранением версий (PyDrive2)
CUSTOM_GDRIVE_FOLDER = "" #@param {type:"string"}
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
import os, re

file_to_upload = ARCHIVE_FILENAME if os.path.exists(ARCHIVE_FILENAME) else "cartridge.omni.zip"

if not os.path.exists(file_to_upload):
    print(f"❌ Ошибка: Локальный файл {file_to_upload} не найден. Сначала выполните шаг 4!")
else:
    # 1. Авторизация в сервисе Google Drive API
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    # 2. Определение целевых папок для загрузки
    default_targets = [
        ("Основная папка Triad-Omni", "1UNP2g3jOfGjxCUDhBDQtMuEFvoESBp1K"),
        ("Общая папка синхронизации", "1WJj716ROzvPpavqRgsLhZ6ik4PfJzKqV")
    ]

    targets = []
    if CUSTOM_GDRIVE_FOLDER and CUSTOM_GDRIVE_FOLDER.strip():
        raw_target = CUSTOM_GDRIVE_FOLDER.strip()
        # Извлекаем ID папки если вставили полную ссылку (например https://drive.google.com/drive/folders/1ABC...)
        match = re.search(r"folders/([a-zA-Z0-9_-]+)", raw_target)
        folder_id = match.group(1) if match else raw_target
        targets.append(("Пользовательская папка Google Диска", folder_id))
        print(f"📁 Выбрана пользовательская папка Google Диска: {folder_id}")
    else:
        targets = default_targets

    for name, folder_id in targets:
        print(f"⏳ Загрузка {file_to_upload} в '{name}' (ID: {folder_id})...")
        try:
            # 1. Загружаем версионированный файл (несгораемый архив истории)
            drive_file = drive.CreateFile({
                "title": os.path.basename(file_to_upload),
                "parents": [{"id": folder_id}],
                "description": f"Triad-Omni LoRA Cartridge: {CARTRIDGE_SLUG} {BUILD_TAG} (Rank {LORA_RANK}, Steps {MAX_STEPS})"
            })
            drive_file.SetContentFile(file_to_upload)
            drive_file.Upload()
            print(f"  ✅ Архив версии загружен: {file_to_upload}")
            print(f"  🔗 Ссылка на файл: {drive_file.get('alternateLink')}")

            # 2. Также загружаем/обновляем latest указатель для быстрого скачивания
            if os.path.exists(LATEST_FILENAME):
                latest_drive = drive.CreateFile({
                    "title": os.path.basename(LATEST_FILENAME),
                    "parents": [{"id": folder_id}],
                    "description": f"Latest build: {BUILD_TAG}"
                })
                latest_drive.SetContentFile(LATEST_FILENAME)
                latest_drive.Upload()
                print(f"  ✅ Указатель свежей версии загружен: {LATEST_FILENAME}")
        except Exception as e:
            print(f"  ❌ Ошибка загрузки в папку {folder_id}: {e}")

    print("🎉 Все версии и архивы успешно сохранены на Google Диске без потери истории!")